# LLZO 快速运行测试（ReaxNet + PQEq）+ 短程 NVT MD

这个 notebook 的目标是：
1. 用 `data/LLZO/init.vasp` 跑通 **LLZO 单点能量/力/电荷**；
2. 展示 `E0 + EPQEq (+ED3)` 在代码里如何拼起来；
3. 用 **ASE Langevin** 做一段 **NVT 分子动力学**，并把轨迹存成 `.traj`。

> 单点部分用于确认环境与预训练模型；MD 部分默认 **步数很少**（1536 原子每步很贵），确认无误后再加大 `MD_STEPS` / 换 `MD_TIMESTEP_FS`。

## 若提示 `No module named 'reaxnet'`

下面第一个代码格已用 **`sys.path.insert`** 把项目根目录加入 Python 搜索路径，一般无需再装包。

若你更习惯「魔法命令」装成可编辑包（与终端里 `pip install -e .` 相同），可在**单独一格**里运行：

```python
%pip install -e /data/home/public/qiuqizhi/reaxnet
```

装好后 `import reaxnet` 会走 site-packages 里的链接，不依赖 `sys.path`。

In [3]:
# 这格做四件事：
# 1) 固定项目根目录（含 setup.py 的那一层，下面有子目录 reaxnet/ 才是 Python 包）
# 2) 把该目录加入 sys.path，这样未 pip install 时也能 import reaxnet
#    （等价于在终端里 cd 到项目根后执行 pip install -e .）
# 3) 导入依赖并开启 float64
# 4) 打印路径，便于排查

import sys
from pathlib import Path

PROJECT_ROOT = Path('/data/home/public/qiuqizhi/reaxnet').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import pickle
import jax
import jax.numpy as jnp
import numpy as np
from functools import partial
from ase.io import read
from jax_md import space, partition

from reaxnet.egnn.nequip import NequIPEnergyModel
from reaxnet.egnn.data import AtomicNumberTable
from reaxnet.egnn.nn_util import neighbor_list_featurizer
from reaxnet.jax_nb.parameters import pqeq_parameters
from reaxnet.jax_nb.jax_nb import pqeq_fori_loop, nonbond_potential, LAMBDA

jax.config.update("jax_enable_x64", True)

LLZO_PATH = PROJECT_ROOT / 'data' / 'LLZO' / 'init.vasp'
MODEL_DIR = PROJECT_ROOT / 'reaxnet' / 'pretrained'

print('PROJECT_ROOT (on sys.path) =', PROJECT_ROOT)
print('LLZO_PATH                  =', LLZO_PATH)
print('MODEL_DIR                  =', MODEL_DIR)

PROJECT_ROOT (on sys.path) = /data/home/public/qiuqizhi/reaxnet
LLZO_PATH                  = /data/home/public/qiuqizhi/reaxnet/data/LLZO/init.vasp
MODEL_DIR                  = /data/home/public/qiuqizhi/reaxnet/reaxnet/pretrained


In [5]:
# 这格做输入与模型加载：
# - 读取 LLZO 结构
# - 读取预训练模型配置/参数/元素映射
# - 创建神经网络模型对象

required_model_files = [
    MODEL_DIR / 'model_config.yaml',
    MODEL_DIR / 'params.pickle',
    MODEL_DIR / 'mapping.yaml',
]

missing = [str(p) for p in required_model_files if not p.exists()]
if not LLZO_PATH.exists():
    raise FileNotFoundError(f'未找到 LLZO 结构文件: {LLZO_PATH}')
if missing:
    raise FileNotFoundError(
        '缺少预训练模型文件，请先按 README/pretrained/README.md 下载后放到 reaxnet/pretrained:\n'
        + '\n'.join(missing)
    )

atoms = read(str(LLZO_PATH))
print('LLZO 原子数 =', len(atoms))
print('元素组成    =', atoms.symbols)

with open(MODEL_DIR / 'model_config.yaml', 'r') as f:
    model_dict = yaml.safe_load(f)
with open(MODEL_DIR / 'params.pickle', 'rb') as f:
    params = pickle.load(f)

ztable = AtomicNumberTable.from_dict(str(MODEL_DIR / 'mapping.yaml'))
model = NequIPEnergyModel(**model_dict)

print('NN cutoff r_max =', model_dict['r_max'])

LLZO 原子数 = 1536
元素组成    = Li448O768La192Zr128
NN cutoff r_max = 5.0


/tmp/ipykernel_4104172/837436127.py:28: DeprecationWarning: Pickled array contains an aval with a named_shape attribute. This is deprecated and the code path supporting such avals will be removed. Please re-pickle the array.
  params = pickle.load(f)


In [6]:
# 这格是核心：把论文里的 E0 + EPQEq (+ED3) 组起来。
# - E0: 等变神经网络短程项
# - EPQEq: 可极化电荷平衡长程静电项
# - ED3: 可选色散项（这里默认关掉，先快速跑通）

# 使用分数坐标 + 周期边界（与官方 examples/basic.ipynb 一致）
positions = jnp.asarray(atoms.get_scaled_positions())
box = jnp.asarray(atoms.get_cell().array.transpose())
atomic_numbers = jnp.asarray(atoms.numbers)
chemical_symbols = atoms.get_chemical_symbols()

nn_atomic_numbers = ztable.mapping(atomic_numbers)
nn_atomic_numbers = jax.nn.one_hot(jnp.array(nn_atomic_numbers), len(ztable) + 1)

displacement_fn, _ = space.periodic_general(box, fractional_coordinates=True)

nn_neighbor_fn = partition.neighbor_list(
    displacement_fn,
    box,
    model_dict['r_max'],
    format=partition.Sparse,
    fractional_coordinates=True,
)

nb_neighbor_fn = partition.neighbor_list(
    displacement_fn,
    box,
    12.5,
    format=partition.Sparse,
    fractional_coordinates=True,
    capacity_multiplier=2.0,
)

featurizer = neighbor_list_featurizer(displacement_fn)

# ---- E0 (NN short-range) ----
def energy_nn(embedded_numbers, model, params, position, neighbor, **kwargs):
    graph = featurizer(embedded_numbers, position, neighbor, **kwargs)
    atomic_output = model.apply(params, graph.edges, graph.nodes, graph.senders, graph.receivers)
    return jnp.sum(atomic_output[:-1])

energy_fn_nn = partial(energy_nn, nn_atomic_numbers, model, params)

# ---- EPQEq 参数准备 ----
rad = jnp.array([pqeq_parameters[s]['rad'] for s in chemical_symbols])
alpha = 0.5 * LAMBDA / rad / rad
alpha = jnp.sqrt(alpha.reshape(-1, 1) * alpha.reshape(1, -1) / (alpha.reshape(-1, 1) + alpha.reshape(1, -1)))
chi0 = jnp.array([pqeq_parameters[s]['chi0'] for s in chemical_symbols])
eta0 = jnp.array([pqeq_parameters[s]['eta0'] for s in chemical_symbols])
z = jnp.array([pqeq_parameters[s]['Z'] for s in chemical_symbols])
Ks = jnp.array([pqeq_parameters[s]['Ks'] for s in chemical_symbols])

charges_fn = partial(
    pqeq_fori_loop,
    displacement_fn,
    alpha=alpha,
    cutoff=12.5,
    iterations=2,
    net_charge=0.0,
    eta0=eta0,
    chi0=chi0,
    z=z,
    Ks=Ks,
)

USE_D3 = False  # 想更贴近论文总能可改 True，但首次测试建议 False 加快速度
energy_fn_nb = partial(
    nonbond_potential,
    displacement_fn,
    alpha=alpha,
    cutoff=12.5,
    eta0=eta0,
    chi0=chi0,
    z=z,
    Ks=Ks,
    compute_d3=USE_D3,
    atomic_numbers=atomic_numbers,
    d3_params={'s6': 1.0, 'rs6': 1.217, 's18': 0.722, 'rs18': 1.0, 'alp': 14.0},
    damping='zero',
    smooth_fn=None,
)


def total_energy_fn(positions, nn_nbr, nb_nbr):
    # 每次调用都更新邻居表
    nn_nbr = nn_nbr.update(positions)
    nb_nbr = nb_nbr.update(positions)

    pe_nn = energy_fn_nn(positions, nn_nbr)
    charges, r_shell = charges_fn(jax.lax.stop_gradient(positions), nb_nbr)
    pe_nb = energy_fn_nb(positions, nb_nbr, r_shell, charges)

    return pe_nn + pe_nb, (charges, r_shell)


# 分配邻居表，并 JIT 编译 value+grad
nn_nbr = nn_neighbor_fn.allocate(positions)
nb_nbr = nb_neighbor_fn.allocate(positions)

value_and_grad_fn = jax.jit(
    jax.value_and_grad(
        partial(total_energy_fn, nn_nbr=nn_nbr, nb_nbr=nb_nbr),
        argnums=0,
        has_aux=True,
    )
)

print('函数已准备完成，可进行 LLZO 单点测试。')

/data/home/public/tzh/anaconda3/envs/qqz/lib/python3.10/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=int32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


函数已准备完成，可进行 LLZO 单点测试。


In [7]:
# 单点运行测试：输出总能、力、部分电荷统计
# 注意：第一次运行会触发 JIT 编译，耗时会明显更长。

results = value_and_grad_fn(positions)

total_energy_eV = float(results[0][0])
forces_eV_A = np.asarray(-results[1])
charges_e = np.asarray(results[0][1][0])

print(f'Total Energy (eV)      : {total_energy_eV:.6f}')
print(f'Energy / atom (eV/atom): {total_energy_eV / len(atoms):.6f}')
print('Forces shape            :', forces_eV_A.shape)
print('Charges shape           :', charges_e.shape)
print(f'Charge sum (should ~0)  : {charges_e.sum():.6e}')
print(f'Charge range            : [{charges_e.min():.4f}, {charges_e.max():.4f}]')

Total Energy (eV)      : -11212.005346
Energy / atom (eV/atom): -7.299483
Forces shape            : (1536, 3)
Charges shape           : (1536,)
Charge sum (should ~0)  : -2.842171e-14
Charge range            : [-0.3862, 0.6220]


## 第二部分：LLZO 短程 NVT 分子动力学（ASE + 同一套势）

下面用 **ASE Langevin** 做恒温 MD（与论文里 Nosé–Hoover 不同，但便于先跑通）。要点：

- `value_and_grad_fn` 对 **分数坐标**求导；ASE 积分用 **笛卡尔坐标**，因此力要按链式法则转换：  
  `F_cart = -(∂E/∂s) @ inv(cell^T)`（与 ASE 约定 `pos_cart = scaled @ cell` 一致）。
- LLZO 约 1536 原子，**每步一次全体系能量+力很贵**，默认只跑少量步数做烟测；正式计算请加大步数或换小体系。
- **NPT**（论文先 NpT 再 NVT）需应力或变胞积分器，本示例未实现；可先 NVT 验证轨迹与能量漂移。

---

## 其他可尝试

- 换 `LLZO_PATH` / 不同 `*_final.vasp` 作初态；`USE_D3 = True` 更接近论文总能。
- 长轨迹写入后可用 `ase.geometry.analysis` 或自写 MSD 做扩散分析。

In [13]:
# ASE 计算器：把当前 atoms 的分数坐标送进 value_and_grad_fn，返回能量与笛卡尔力。
# forces: F = -∂E/∂r_cart ；由 ∂E/∂s（s 为 scaled）得到 ∂E/∂r：∂E/∂s = ∂E/∂r @ cell^T  =>  ∂E/∂r = ∂E/∂s @ inv(cell^T)

from ase.calculators.calculator import Calculator, all_changes
from ase import units


class ReaxNetPQEqCalculator(Calculator):
    implemented_properties = ['energy', 'forces']

    def __init__(self, vg_fn):
        super().__init__()
        self.vg_fn = vg_fn

    def calculate(self, atoms, properties, system_changes):
        if len(system_changes) == 0:
            system_changes = all_changes
        scaled = atoms.get_scaled_positions(wrap=True)
        scaled_j = jnp.asarray(scaled)
        (energy, _aux), grad_frac = self.vg_fn(scaled_j)
        grad_frac = np.asarray(grad_frac)
        cell = atoms.get_cell().array  # rows = lattice vectors, pos_cart = scaled @ cell
        inv_cell_T = np.linalg.inv(cell.T)
        forces = -grad_frac @ inv_cell_T
        self.results = {'energy': float(energy), 'forces': forces}


# ---------- 三套 MD 参数模板（测试档 / 预跑档 / 正式档）----------
# 切换方式：把 MD_PROFILE 改为 'test' / 'prerun' / 'production'
MD_PROFILE = 'prerun'

MD_PROFILES = {
    # 仅做代码与数值烟测：几秒到几分钟（取决于机器）
    'test': {
        'temp_k': 1200.0,
        'timestep_fs': 1.0,
        'steps': 20,
        'friction': 0.02,
        'traj_name': 'llzo_nvt_test.traj',
    },
    # 预跑：检查温度稳定性/能量走势/轨迹输出是否正常
    'prerun': {
        'temp_k': 1200.0,
        'timestep_fs': 1.0,
        'steps': 2000,
        'friction': 0.02,
        'traj_name': 'llzo_nvt_prerun.traj',
    },
    # 正式档（接近论文量级中的“生产段”设定思路；时间成本很高）
    'production': {
        'temp_k': 1200.0,
        'timestep_fs': 2.0,
        'steps': 1000000,
        'friction': 0.02,
        'traj_name': 'llzo_nvt_production.traj',
    },
}

if MD_PROFILE not in MD_PROFILES:
    raise ValueError(f"未知 MD_PROFILE={MD_PROFILE}，可选: {list(MD_PROFILES.keys())}")

cfg = MD_PROFILES[MD_PROFILE]
MD_TEMP_K = cfg['temp_k']
MD_TIMESTEP_FS = cfg['timestep_fs']
MD_STEPS = cfg['steps']
MD_FRICTION = cfg['friction']
TRAJ_PATH = str(PROJECT_ROOT / 'my_experiment' / cfg['traj_name'])
RNG_SEED = 42

atoms_md = atoms.copy()
atoms_md.calc = ReaxNetPQEqCalculator(value_and_grad_fn)

print('当前档位:', MD_PROFILE)
print('MD 将写入:', TRAJ_PATH)
print('TEMP_K =', MD_TEMP_K, 'TIMESTEP_FS =', MD_TIMESTEP_FS, 'STEPS =', MD_STEPS, 'FRICTION =', MD_FRICTION)

当前档位: prerun
MD 将写入: /data/home/public/qiuqizhi/reaxnet/my_experiment/llzo_nvt_prerun.traj
TEMP_K = 1200.0 TIMESTEP_FS = 1.0 STEPS = 2000 FRICTION = 0.02


In [14]:
# 运行 Langevin NVT（会每步调用一次势能与力，1536 原子时很慢）

from ase.md.langevin import Langevin
from ase.io import Trajectory
from ase.constraints import FixCom

np.random.seed(RNG_SEED)

# 按 ASE 新建议：fixcm=False + FixCom 约束
atoms_md.set_constraint(FixCom())

dyn = Langevin(
    atoms_md,
    timestep=MD_TIMESTEP_FS * units.fs,
    temperature_K=MD_TEMP_K,
    friction=MD_FRICTION,
    fixcm=False,
    logfile=str(PROJECT_ROOT / 'my_experiment' / f'llzo_md_{MD_PROFILE}.log'),
)

traj = Trajectory(TRAJ_PATH, mode='w', atoms=atoms_md)
traj.write(atoms_md)  # 写入初态，便于与后续帧对齐

def append_traj():
    traj.write(atoms_md)

dyn.attach(append_traj, interval=1)

dyn.run(MD_STEPS)
traj.close()

print('MD 结束，轨迹帧数约 =', MD_STEPS + 1, '（含初态请用 Trajectory 读取确认）')

MD 结束，轨迹帧数约 = 2001 （含初态请用 Trajectory 读取确认）


In [10]:
# 可选：读回轨迹，看最后一帧能量（再算一次单点，验证与 MD 末态一致）

traj_r = Trajectory(TRAJ_PATH, mode='r')
last = traj_r[-1]
traj_r.close()

calc_tmp = ReaxNetPQEqCalculator(value_and_grad_fn)
last.calc = calc_tmp
print('最后一帧单点能量 (eV):', last.get_potential_energy())
print('最后一帧原子数:', len(last))

最后一帧单点能量 (eV): -11239.098662856868
最后一帧原子数: 1536
